# ChatPromptTemplate

1 实例化的方式 （两种方法：使用构造方法、 from_messages()

2 调用提示词的方式（format()、invoke()、format_messages() format_prompt()）

3 更丰富的实例化参数类型

4 结合LLM

5 插入消息列表MessagePlaceholder

In [1]:
from functools import partial

from langchain_core.prompts import ChatPromptTemplate

## 使用构造方法实例化


prompt_template = ChatPromptTemplate(
    messages=[
        ("system", "你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格"),
        ("human", "{input}"),
    ],
    # input_variables=["input"],
)
prompt = prompt_template.invoke(input={"input","今天真是一个美好的一天"})
print(prompt)
print(type(prompt))

messages=[SystemMessage(content='你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格', additional_kwargs={}, response_metadata={}), HumanMessage(content="{'input', '今天真是一个美好的一天'}", additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


In [10]:
## from_messages() 方法实例化

prompt_template = ChatPromptTemplate.from_messages(messages=[
    ("system", "你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格"),
    ("human", "翻译这句话：{input}"),
])


In [11]:
## 调用提示词模板的几种方式

### invoke() 传入字典，返回ChatPromptValue
### format() 传入变量的值，返回字符串

prompt = prompt_template.format(input="今天真是一个美好的一天")
print(prompt)


System: 你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格
Human: 翻译这句话：今天真是一个美好的一天


In [12]:
### format_messages() 传入变量的值，返回消息列表list
prompt1 = prompt_template.format_messages(input="该死")
print(prompt1)
print(type(prompt1))

[SystemMessage(content='你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格', additional_kwargs={}, response_metadata={}), HumanMessage(content='翻译这句话：该死', additional_kwargs={}, response_metadata={})]
<class 'list'>


In [13]:
### format_prompt() 传入变量的值，返回ChatPromptValue
prompt = prompt_template.format_prompt(input="今天真是一个美好的一天")
print(prompt)
print(type(prompt))


messages=[SystemMessage(content='你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格', additional_kwargs={}, response_metadata={}), HumanMessage(content='翻译这句话：今天真是一个美好的一天', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>


In [14]:
### 如何实现ChatPromptValue list[messages] 和 str 之间的转换
from langchain_core.prompt_values import ChatPromptValue

prompt = prompt_template.format_messages(input="今天真是一个美好的一天")
print(prompt)
print(type(prompt))
chat_prompt_value = ChatPromptValue(messages=prompt)
print(chat_prompt_value)
print(type(chat_prompt_value))

### 如何实现ChatPromptValue 和 str 之间的转换
str_prompt = chat_prompt_value.to_string()
print(str)
print(str(chat_prompt_value))




<class 'list'>
messages=[SystemMessage(content='你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格', additional_kwargs={}, response_metadata={}), HumanMessage(content='翻译这句话：今天真是一个美好的一天', additional_kwargs={}, response_metadata={})]
<class 'langchain_core.prompt_values.ChatPromptValue'>
<class 'str'>
messages=[SystemMessage(content='你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格', additional_kwargs={}, response_metadata={}), HumanMessage(content='翻译这句话：今天真是一个美好的一天', additional_kwargs={}, response_metadata={})]


In [15]:
## 更丰富的实例化参数类型
### 参数是列表类型，列表的元素可以是字符串、字典、字符串构成的元组、消息类型、提示词模板类型、消息提示词模板类型等

### str
prompt_template = ChatPromptTemplate.from_messages(["我的问题是{question}"]) # 默认是human
prompt = prompt_template.invoke({"question":"你好"})
print(prompt)

messages=[HumanMessage(content='我的问题是你好', additional_kwargs={}, response_metadata={})]


In [16]:
### 字典
prompt_template =  ChatPromptTemplate.from_messages([{"role":"user","content":"你好"},{"role":"assistant","content":"你好！有什么我可以帮助你{temp}？"}])
prompt = prompt_template.format(temp="吗？")
print(prompt)

Human: 你好
AI: 你好！有什么我可以帮助你吗？？


In [17]:
from langchain_core.messages import SystemMessage, HumanMessage

### 消息类型
prompt_template = ChatPromptTemplate.from_messages([
    SystemMessage(content="你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格"),
    HumanMessage(content="翻译这句话{sentence}")
])
###SystemMessage 和 HumanMessage 是已经创建好的消息对象。它们的 content 属性是固定的字符串，不会再进行模板替换，ChatPromptTemplate 只会对元组形式的消息进行模板替换，而不会处理已创建的消息对象

prompt = prompt_template.invoke({"sentence":"你好"})
print(prompt)

messages=[SystemMessage(content='你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格', additional_kwargs={}, response_metadata={}), HumanMessage(content='翻译这句话{sentence}', additional_kwargs={}, response_metadata={})]


In [18]:
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv(override=True)
print(os.getenv("MODELSPACE_MODEL"))

chat_model = ChatOpenAI(
    model=os.getenv("MODELSPACE_MODEL"), # 默认是gpt-3.5-turbo
    api_key=os.getenv("MODELSPACE_API_KEY"),
    base_url=os.getenv("MODELSPACE_BASE_URL"),
    temperature=0.7,
    # streaming=True # 开启流式调用
)

prompt_template = ChatPromptTemplate.from_messages(messages=[
    ("system", "你是一个专业中译英的翻译，具有强烈的美国西海岸黑人的风格"),
    ("human", "翻译这句话：{input}"),
])

prompt1 = prompt_template.format_messages(input="滚远点")

response = chat_model.invoke(prompt1)
print(response.content)


Back up off me, yo.


In [19]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage

prompt_template = ChatPromptTemplate([
    ("system", "You are a helpful assistant,我能帮你解释{area}方面的知识"),
    MessagesPlaceholder("msgs")
],partial_variables={"area":"天文学"},input_variables=["msgs"])
# prompt_template.invoke({"msgs": [HumanMessage(content="hi!")]})

prompt_template = prompt_template.format_messages(msgs=[HumanMessage(content="hi!")])
# prompt_template = prompt_template.format_messages(msgs="我是一个消息")  #错误的
# prompt_template = prompt_template.format_messages(msgs=[
#     SystemMessage(content="我的名字叫小智"),
#     HumanMessage(content="帮我解释一下为什么星星会一闪一闪的？")
# ],area="天文学")

print(prompt_template)

[SystemMessage(content='You are a helpful assistant,我能帮你解释天文学方面的知识', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi!', additional_kwargs={}, response_metadata={})]
